# School Teacher Agent

An agentic system that helps teachers by looking up curriculum standards,
generating lesson plans, creating quizzes, grading student responses, and
adapting difficulty — all orchestrated via OpenAI function calling.

## Why an Agent?

Lesson planning is **iterative and context-dependent**: a teacher says "I need
a 45-minute lesson on fractions for 4th graders" and the system must look up
the relevant Common Core standards, generate a lesson plan aligned to those
standards, create an assessment, and potentially adjust difficulty based on
how sample students perform. An agent can handle this multi-step workflow
naturally, deciding whether to generate a quiz *before* or *after* seeing
the lesson plan, and looping back to adjust when grading reveals that the
difficulty was too high or too low.

In [ ]:
%pip install openai requests pandas --quiet

In [ ]:
import json, textwrap
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## Common Core Standards Reference

A subset of Common Core State Standards embedded as a dictionary for quick
lookup by the agent. In production this would come from an API or database.

In [ ]:
STANDARDS = {
    "math": {
        3: {
            "3.NF.A.1": "Understand a fraction 1/b as the quantity formed by 1 part when a whole is partitioned into b equal parts.",
            "3.NF.A.2": "Understand a fraction as a number on the number line.",
            "3.OA.A.1": "Interpret products of whole numbers, e.g., interpret 5 x 7 as the total number of objects in 5 groups of 7.",
            "3.OA.C.7": "Fluently multiply and divide within 100.",
            "3.MD.B.3": "Draw a scaled picture graph and a scaled bar graph to represent a data set.",
        },
        4: {
            "4.NF.A.1": "Explain why a fraction a/b is equivalent to a fraction (n x a)/(n x b).",
            "4.NF.A.2": "Compare two fractions with different numerators and different denominators.",
            "4.NF.B.3": "Understand a fraction a/b with a > 1 as a sum of fractions 1/b.",
            "4.OA.A.1": "Interpret a multiplication equation as a comparison.",
            "4.MD.A.1": "Know relative sizes of measurement units within one system of units.",
        },
        5: {
            "5.NF.A.1": "Add and subtract fractions with unlike denominators.",
            "5.NF.B.4": "Apply and extend previous understandings of multiplication to multiply a fraction by a whole number.",
            "5.NBT.A.1": "Recognize that in a multi-digit number, a digit in one place represents 10 times as much as it represents in the place to its right.",
            "5.OA.B.3": "Generate two numerical patterns using two given rules. Identify apparent relationships between corresponding terms.",
        },
    },
    "ela": {
        3: {
            "RL.3.1": "Ask and answer questions to demonstrate understanding of a text, referring explicitly to the text.",
            "RL.3.3": "Describe characters in a story and explain how their actions contribute to the sequence of events.",
            "W.3.1": "Write opinion pieces on topics or texts, supporting a point of view with reasons.",
        },
        4: {
            "RL.4.1": "Refer to details and examples in a text when explaining what the text says explicitly and when drawing inferences.",
            "RL.4.2": "Determine a theme of a story, drama, or poem from details in the text; summarize the text.",
            "W.4.1": "Write opinion pieces on topics or texts, supporting a point of view with reasons and information.",
            "W.4.3": "Write narratives to develop real or imagined experiences using effective technique, descriptive details, and clear event sequences.",
        },
        5: {
            "RL.5.2": "Determine a theme of a story, drama, or poem from details in the text, including how characters respond to challenges.",
            "RI.5.5": "Compare and contrast the overall structure of events, ideas, concepts, or information in two or more texts.",
            "W.5.2": "Write informative/explanatory texts to examine a topic and convey ideas and information clearly.",
        },
    },
}

# OpenStax textbook references for enrichment
OPENSTAX_REFS = {
    "fractions": "OpenStax Elementary Algebra 2e, Chapter 4: Fractions (https://openstax.org/details/books/elementary-algebra-2e)",
    "multiplication": "OpenStax Prealgebra 2e, Chapter 3: Multiplication and Division (https://openstax.org/details/books/prealgebra-2e)",
    "reading_comprehension": "OpenStax Writing Guide, Chapter 6: Reading Critically (https://openstax.org/details/books/writing-guide)",
    "narrative_writing": "OpenStax Writing Guide, Chapter 13: Narrative (https://openstax.org/details/books/writing-guide)",
}

## Tool 1 — `get_standards`

Looks up Common Core standards for a given grade and subject.

In [ ]:
def get_standards(grade: int, subject: str) -> str:
    """Return Common Core standards for a grade (3-5) and subject ('math' or 'ela')."""
    subj = subject.lower()
    if subj not in STANDARDS:
        return json.dumps({"error": f"Subject '{subject}' not found. Available: math, ela"})
    grade_standards = STANDARDS[subj].get(grade)
    if not grade_standards:
        return json.dumps({"error": f"Grade {grade} not found for {subject}. Available: 3, 4, 5"})

    # Also include relevant OpenStax references
    refs = {k: v for k, v in OPENSTAX_REFS.items()}

    return json.dumps(
        {"grade": grade, "subject": subj, "standards": grade_standards, "openstax_references": refs},
        indent=2,
    )

## Tool 2 — `generate_lesson_plan`

Uses the LLM to generate a structured lesson plan. This is a **tool that
itself calls the LLM** — a common pattern in agentic systems where one tool
delegates complex generation to the model.

In [ ]:
def generate_lesson_plan(topic: str, grade: int, duration: int = 45) -> str:
    """Generate a structured lesson plan for a topic, grade level, and duration in minutes."""
    prompt = textwrap.dedent(f"""\
        Create a detailed lesson plan with this structure:
        - Title
        - Grade Level: {grade}
        - Duration: {duration} minutes
        - Topic: {topic}
        - Learning Objectives (2-3 bullet points)
        - Materials Needed
        - Lesson Phases:
          1. Warm-Up / Hook (5 min)
          2. Direct Instruction (10-15 min)
          3. Guided Practice (10-15 min)
          4. Independent Practice (10 min)
          5. Closure / Exit Ticket (5 min)
        - Differentiation (for struggling and advanced learners)

        Return the plan as a JSON object with these keys:
        title, grade, duration_minutes, topic, objectives (list),
        materials (list), phases (list of {{name, duration_minutes, description}}),
        differentiation ({{struggling, advanced}})
    """)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

## Tool 3 — `create_quiz`

Generates a quiz with questions, answer choices, correct answers, and rubrics.

In [ ]:
def create_quiz(topic: str, difficulty: str = "medium", num_questions: int = 5) -> str:
    """Create a quiz on a topic. Difficulty: 'easy', 'medium', 'hard'.
    Returns questions with answer choices, correct answer, and rubric."""
    prompt = textwrap.dedent(f"""\
        Create a {num_questions}-question quiz on "{topic}" at {difficulty} difficulty.

        Return JSON with key "questions" containing a list. Each question has:
        - question_number (int)
        - question_text (str)
        - question_type: "multiple_choice" or "short_answer"
        - choices (list of str, only for multiple_choice; use A, B, C, D labels)
        - correct_answer (str)
        - rubric (str — what to look for when grading)
        - points (int)

        Mix question types. Total points should be {num_questions * 10}.
    """)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

## Tool 4 — `grade_response`

Uses the LLM to evaluate a student's answer against the rubric and provide
feedback.

In [ ]:
def grade_response(question: str, student_answer: str, rubric: str) -> str:
    """Grade a student's answer to a question using the provided rubric.
    Returns score, feedback, and whether the student needs more practice."""
    prompt = textwrap.dedent(f"""\
        You are grading a student response.

        Question: {question}
        Student Answer: {student_answer}
        Rubric: {rubric}

        Return JSON with:
        - score (int, 0-10)
        - max_score (int, 10)
        - is_correct (bool)
        - feedback (str — encouraging and specific)
        - needs_practice (bool — true if score < 7)
        - misconception (str or null — describe any misconception detected)
    """)

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

## Tool Schema for OpenAI Function Calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_standards",
            "description": "Look up Common Core standards for a grade (3-5) and subject (math or ela).",
            "parameters": {
                "type": "object",
                "properties": {
                    "grade": {"type": "integer", "description": "Grade level (3, 4, or 5)"},
                    "subject": {"type": "string", "enum": ["math", "ela"], "description": "Subject area"},
                },
                "required": ["grade", "subject"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "generate_lesson_plan",
            "description": "Generate a structured lesson plan for a topic, grade, and duration.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "Lesson topic"},
                    "grade": {"type": "integer", "description": "Grade level"},
                    "duration": {"type": "integer", "description": "Duration in minutes (default 45)"},
                },
                "required": ["topic", "grade"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_quiz",
            "description": "Create a quiz on a topic with specified difficulty and number of questions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "Quiz topic"},
                    "difficulty": {"type": "string", "enum": ["easy", "medium", "hard"], "description": "Difficulty level"},
                    "num_questions": {"type": "integer", "description": "Number of questions (default 5)"},
                },
                "required": ["topic"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "grade_response",
            "description": "Grade a student's answer using the provided rubric. Returns score, feedback, and whether more practice is needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The question text"},
                    "student_answer": {"type": "string", "description": "The student's answer"},
                    "rubric": {"type": "string", "description": "Grading rubric"},
                },
                "required": ["question", "student_answer", "rubric"],
            },
        },
    }
]

## Dispatcher

In [ ]:
TOOL_MAP = {
    "get_standards": get_standards,
    "generate_lesson_plan": generate_lesson_plan,
    "create_quiz": create_quiz,
    "grade_response": grade_response,
}

## Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a Teaching Assistant agent. Your job is to help teachers
    plan lessons, create assessments, and analyze student performance.

    Workflow:
    1. When a teacher requests a lesson, first look up the relevant
       Common Core standards using get_standards.
    2. Generate a lesson plan aligned to those standards.
    3. Create a quiz to assess understanding.
    4. If sample student responses are provided, grade them.
    5. If grading reveals students are struggling, suggest adapting
       difficulty (create an easier quiz) or providing remediation.

    Always reference specific standard IDs (e.g., 4.NF.A.1) in your
    final response. Be encouraging and practical.
""")


def run_agent(user_message: str, max_turns: int = 15) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"\n--- Agent turn {turn + 1} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print("\n--- Agent finished ---")
            return msg.content

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  Calling {fn_name}({json.dumps(fn_args)[:100]})")
            fn = TOOL_MAP.get(fn_name)
            result = fn(**fn_args) if fn else json.dumps({"error": f"Unknown tool: {fn_name}"})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Agent reached maximum turns without finishing."

## Example Execution — Fractions Lesson for 4th Grade

The teacher asks for a complete lesson + quiz package.

In [ ]:
report = run_agent(
    "I'm a 4th grade math teacher. I need a 45-minute lesson on comparing "
    "fractions with different denominators. Please look up the relevant "
    "standards, generate a lesson plan, and create a 5-question quiz at "
    "medium difficulty. Then grade these sample student responses:\n"
    "- Q1 (Which is larger, 3/4 or 2/3?): Student said '3/4 because 4 is bigger than 3'\n"
    "- Q2 (Compare 1/2 and 3/8): Student said '3/8 because 3 and 8 are both bigger'\n"
    "Based on the grading, tell me if I should adjust the difficulty."
)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Second Scenario — ELA Narrative Writing for 5th Grade

In [ ]:
report2 = run_agent(
    "I teach 5th grade ELA. I need a 50-minute lesson on narrative writing — "
    "specifically, developing characters through their actions and dialogue. "
    "Look up the standards, create the lesson plan, make a 4-question quiz "
    "(easy difficulty), and grade this sample response:\n"
    "- Q: 'Write a short paragraph where a character's actions show they are brave.'\n"
    "  Student wrote: 'The girl walked into the dark cave even though she was scared. "
    "She held her flashlight tight and kept going.'\n"
    "Summarize everything for me."
)

In [ ]:
Markdown(report2)

## Results Analysis

| Aspect | Detail |
|--------|--------|
| **Standards alignment** | Agent looked up CCSS standards before generating content, ensuring curriculum alignment |
| **Adaptive difficulty** | After grading revealed a misconception (comparing numerators/denominators incorrectly), the agent recommended easier material |
| **Tool-as-LLM pattern** | `generate_lesson_plan`, `create_quiz`, and `grade_response` all delegate to the LLM — the outer agent orchestrates them |
| **Limitations** | Standards DB only covers grades 3-5 and two subjects; grading is LLM-based (not always consistent) |

### Potential Extensions

1. **Full CCSS database** — integrate the complete Common Core API for K-12.
2. **Student profiles** — add a tool to track individual student progress over time.
3. **Worksheet generation** — add a tool that creates printable PDF worksheets.
4. **Parent communication** — generate progress reports and parent-friendly summaries.

---
*Notebook by the LLM Course — Agentic Designs series*